In [0]:
df = spark.table("workspace.default.model_data_2015_2022")

print(f"Rows: {df.count():,}")
print(f"Columns: {len(df.columns)}")

display(df.limit(10))

Rows: 397,423
Columns: 31


credit_score,first_payment_date,first_time_homebuyer,maturity_date,msa,mi_percent,num_units,occupancy_status,cltv,dti,original_upb,ltv,original_interest_rate,channel,prepayment_penalty_flag,amortization_type,property_state,property_type,postal_code,loan_id,loan_purpose,original_loan_term,num_borrowers,seller_name,super_conforming_flag,special_eligibility_program,harp_indicator,property_valuation_method,interest_only_indicator,origination_vintage,seriously_delinquent_24m
734,201504,N,203003,null,0,1,P,80,15,417000,65,2.5,R,N,FRM,CO,SF,812,F15Q10000025,P,180,2,OTHER,N,null,N,null,N,2015,false
797,201504,N,204503,null,0,1,S,37,26,110000,37,3.625,R,N,FRM,NC,PU,285,F15Q10000031,P,360,2,OTHER,N,null,N,null,N,2015,false
670,201504,N,203003,11460.0,12,1,P,90,33,297000,90,3.375,R,N,FRM,MI,SF,481,F15Q10000084,N,180,2,OTHER,N,null,N,null,N,2015,false
731,201503,N,204502,null,0,1,I,73,null,215000,73,4.0,R,N,FRM,IL,SF,601,F15Q10000087,N,360,2,OTHER,N,null,Y,null,N,2015,false
668,201504,N,203003,39340.0,0,2,I,75,30,180000,75,3.75,C,N,FRM,UT,SF,846,F15Q10000157,N,180,2,OTHER,N,null,N,null,N,2015,false
790,201503,Y,204502,20260.0,35,1,P,95,29,150000,95,4.25,R,N,FRM,WI,MH,548,F15Q10000167,P,360,2,OTHER,N,null,N,null,N,2015,false
721,201504,N,204503,null,25,1,P,90,31,95000,90,4.125,R,N,FRM,OH,SF,451,F15Q10000188,P,360,2,OTHER,N,null,N,null,N,2015,false
764,201503,Y,204502,null,0,1,P,80,41,120000,80,3.875,R,N,FRM,ND,SF,584,F15Q10000230,P,360,1,OTHER,N,null,N,null,N,2015,false
748,201503,N,203002,null,0,1,P,69,25,215000,69,3.0,R,N,FRM,IL,SF,619,F15Q10000270,N,180,2,OTHER,N,null,N,null,N,2015,false
771,201503,N,203002,47664.0,0,1,P,80,10,417000,80,3.25,R,N,FRM,MI,SF,483,F15Q10000275,C,180,2,OTHER,N,null,N,null,N,2015,false


In [0]:
from pyspark.sql import functions as F

vintage_summary = (
    df.groupBy("origination_vintage")
      .agg(
          F.count("*").alias("loans"),
          F.sum(F.col("seriously_delinquent_24m").cast("int")).alias("serious_delinquencies"),
          F.avg(F.col("seriously_delinquent_24m").cast("int")).alias("delinquency_rate")
      )
      .orderBy("origination_vintage")
)

display(vintage_summary)

origination_vintage,loans,serious_delinquencies,delinquency_rate
2015,48626,250,0.005141282441492206
2016,49613,329,0.0066313264668534455
2017,49841,395,0.007925202142814149
2018,49908,1274,0.02552696962410836
2019,49897,2285,0.04579433633284566
2020,49877,791,0.015859013172404113
2021,49895,445,0.008918729331596353
2022,49766,882,0.017722943374994977


In [0]:
from pyspark.sql import functions as F

risk_summary = (
    df.withColumn(
        "fico_group",
        F.when(F.col("credit_score") < 650, "<650")
         .when(F.col("credit_score") < 700, "650-699")
         .when(F.col("credit_score") < 750, "700-749")
         .when(F.col("credit_score") < 800, "750-799")
         .otherwise("800+")
    )
    .groupBy("fico_group")
    .agg(
        F.count("*").alias("loans"),
        F.avg(F.col("seriously_delinquent_24m").cast("int"))
         .alias("delinquency_rate")
    )
    .orderBy("fico_group")
)

display(risk_summary)

fico_group,loans,delinquency_rate
650-699,52730,0.03559643466717239
700-749,112181,0.021073087242937754
750-799,171819,0.008910539579441157
800+,49284,0.004261017774531288
<650,11409,0.058637917433605045
